In [113]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from tqdm.auto import tqdm
from pathlib import Path

In [114]:
src_dir = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/04_Data/2026_data/"
)
excel_path = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/data_correlation.xlsx"
)


print (excel_path)


C:\Users\hermawan\OneDrive - Stichting Deltares\PhD\Egypt\data_correlation.xlsx


In [115]:
survey_df = pd.read_csv(src_dir.parent / "Result/ind_survey_data_gov_assigned.csv")
distribution_df = pd.read_csv(
    src_dir.parent / "Result/population_density_matrix.csv", index_col="NAME1_"
)
masking_df = pd.read_excel(
    src_dir.parent / "Result/correlation.xlsx", sheet_name="Sheet1"
)
excel_df = pd.read_excel(excel_path, sheet_name="command_unit")

In [116]:
print(f"Average survey weight: {survey_df['pweight'].mean()}")
print(f"Std of survey weight: {survey_df['pweight'].std()}")

Average survey weight: 341.13162810546737
Std of survey weight: 58.55064178032295


In [117]:
survey_df['sempinc'] = survey_df['sempinc'].fillna(0)
survey_df['irrgwag'] = survey_df['irrgwag'].fillna(0)
survey_df['totwag'] = survey_df['totwag'].fillna(0)

survey_df['totwag'] = survey_df['totwag'] + survey_df['sempinc'] + survey_df['irrgwag'] * 30

In [118]:
female_df = survey_df[survey_df['sex_label'] == 'Female']
male_df = survey_df[survey_df['sex_label'] == 'Male']

In [119]:
governorates = survey_df[survey_df["NAME1_"].notna()]["NAME1_"].unique()

rng = np.random.default_rng(seed=42)
sample_df = survey_df.copy(deep=True)
sample_df["command_unit"] = None
command_units = distribution_df.columns

gov_indices = {
    gov: idx.to_numpy() for gov, idx in survey_df.groupby("NAME1_").groups.items()
}
female_gov_indices = {
    gov: idx.to_numpy() for gov, idx in female_df.groupby("NAME1_").groups.items()
}
male_gov_indices = {
    gov: idx.to_numpy() for gov, idx in male_df.groupby("NAME1_").groups.items()
}

for gov, idx in gov_indices.items():
    probs = distribution_df.loc[gov].to_numpy()

    assignments = rng.choice(
        command_units,
        size=len(idx),
        p=probs,
    )

    sample_df.loc[idx, "command_unit"] = assignments

sample_df.loc[sample_df["command_unit"] == "other", "command_unit"] = None

sample_df = sample_df[sample_df["command_unit"].notna()]
final_sample = sample_df.merge(
    excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
)

to_correlate = masking_df.loc[masking_df["correl"] == 1, "variable"]
cols = [c for c in to_correlate if c in final_sample.columns]

In [120]:
female_df = female_df[[col for col in cols + ["NAME1_", "pweight" ] if col in female_df.columns]]

In [121]:
#process categorical data

#marital. 1=married 0=single
female_df["mart_d"] = female_df["mart_d"].between(
    200, 300, inclusive="both"
).astype(int)

#empstab 1=stable 0=not
female_df["empstab"] = np.where(
    female_df["empstab"] == 1,
    1,
    0
)

#migrant 1=yes 0=no
female_df["immigr"] = female_df["immigr"].fillna(3)
female_df["immigr"] = np.where(
    female_df["immigr"] == 3,
    1,
    0
)
#emps 1=paid 0=unpaid
female_df["emps"] = np.where(
    (female_df["emps"] >= 5) & (female_df["emps"] != 6),
    1,
    0
)


In [122]:
v = female_df.columns.difference(["NAME1_", "pweight"])
female_df[v] = female_df[v].apply(pd.to_numeric, errors="coerce")
female_df["pweight"] = pd.to_numeric(female_df["pweight"], errors="coerce")


female_df = female_df.groupby("NAME1_").apply(
    lambda g: g[v].apply(lambda x: np.average(x, weights=g["pweight"])))


In [123]:
excel_df = excel_df[[col for col in cols + ["area", "arable_km2", "rural_population"]  if col in excel_df.columns]]

REPO_ROOT = Path.cwd().parent

CROSSWALK_CSV = REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "command_area_governorate_crosswalk.csv"
crosswalk_df = pd.read_csv(CROSSWALK_CSV)
crosswalk_df.head()


excel_df["_id"] = excel_df["area"].str.extract(r"(\d+)$")[0].astype("Int64")

excel_df = excel_df.merge(
    crosswalk_df[
        ["command_area_objectid", "gov_NAME1_", "pct_of_governorate_area", "pct_of_command_area_area", "command_area_name"]
    ],
    left_on="_id",
    right_on="command_area_objectid",
    how="left"
).drop(columns=["_id", "command_area_objectid"])


excel_df.head()



,water_productivity,surface_water_mean,ground_water_mean,supply_demand_ratio,water_supply,water_demand,gdp_pc,rwi_median,precipitation,potentialevapotransporation,...,fraction_large,arable_km2_scaled,producer_price,area,arable_km2,rural_population,gov_NAME1_,pct_of_governorate_area,pct_of_command_area_area,command_area_name
0,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,0.0,0.476322,1.552409e+07,Blk_Air_1,178.828424,17108.839844,Behera,0.032411,0.990245,الرشيدية_ومباشر غرب رشيد
1,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,0.0,0.476322,1.552409e+07,Blk_Air_1,178.828424,17108.839844,Kafr-El-Sheikh,0.000381,0.003303,الرشيدية_ومباشر غرب رشيد
2,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Giza,0.001467,0.041068,الرياح البحيرى
3,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Behera,0.113246,0.936307,الرياح البحيرى
4,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Kafr-El-Sheikh,0.000087,0.000203,الرياح البحيرى


In [124]:
#remove production as it seems incorrect
excel_df = excel_df.loc[:, ~excel_df.columns.str.startswith(("salinity"))]
excel_df.head()

,water_productivity,surface_water_mean,ground_water_mean,supply_demand_ratio,water_supply,water_demand,gdp_pc,rwi_median,precipitation,potentialevapotransporation,...,fraction_large,arable_km2_scaled,producer_price,area,arable_km2,rural_population,gov_NAME1_,pct_of_governorate_area,pct_of_command_area_area,command_area_name
0,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,0.0,0.476322,1.552409e+07,Blk_Air_1,178.828424,17108.839844,Behera,0.032411,0.990245,الرشيدية_ومباشر غرب رشيد
1,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,0.0,0.476322,1.552409e+07,Blk_Air_1,178.828424,17108.839844,Kafr-El-Sheikh,0.000381,0.003303,الرشيدية_ومباشر غرب رشيد
2,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Giza,0.001467,0.041068,الرياح البحيرى
3,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Behera,0.113246,0.936307,الرياح البحيرى
4,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,0.0,0.477117,4.934912e+06,Blk_Air_2,661.945281,45841.378906,Kafr-El-Sheikh,0.000087,0.000203,الرياح البحيرى


In [125]:
#get production from iDT
PRODUCTION_CSV = REPO_ROOT / "ERF_Data/Data" / "yield.csv"
production_df = pd.read_csv(PRODUCTION_CSV)
production_df = production_df[production_df["year"] == 2020]

crops = [
    "Rice, paddy",
    "Alfalfa for forage",
    "Maize (incl. 0067 and 0068)",
    "Sugar cane",
    "Wheat"
]

production_df = production_df[
    production_df["crop_name_fao"].str.strip().isin(crops)
]

production_df = production_df.drop(
    columns=["a", "b", "year", "comment", "crop_name", "Unnamed: 0"],
    errors="ignore"
)
production_df.head()

,area_map_name,crop_name_fao,salinity,yield,hectares,corrected_yield,corrected_yield_pp
60,الرشيدية_ومباشر غرب رشيد,Alfalfa for forage,2.324118,3.055946e+08,3525.5088,2.983641e+08,3.349634e+08
61,الرياح البحيرى,Alfalfa for forage,7.360748,1.142664e+09,13182.4040,1.002496e+09,1.125469e+09
62,الخيام,Alfalfa for forage,3.555282,3.163102e+08,3649.1300,2.803977e+08,3.147932e+08
63,نجع حمادى الغربية,Alfalfa for forage,3.522954,2.489504e+09,28720.2900,2.212731e+09,2.484160e+09
64,نجع حمادى الشرقية,Alfalfa for forage,3.460733,7.337176e+08,8464.5730,6.554788e+08,7.358842e+08


In [126]:
base = production_df.groupby("area_map_name").agg(
    salinity=("salinity", "mean"),
    corrected_yield_pp=("corrected_yield_pp", "sum")
)

crop = production_df.pivot_table(
    index="area_map_name",
    columns="crop_name_fao",
    values=["hectares", "corrected_yield"],
    aggfunc="sum",
    fill_value=0
)

crop.columns = [f"{variable}_{crop_name}" for variable, crop_name in crop.columns]

result_production = base.join(crop)

result_production.head()

,salinity,corrected_yield_pp,corrected_yield_Alfalfa for forage,corrected_yield_Maize (incl. 0067 and 0068),"corrected_yield_Rice, paddy",corrected_yield_Sugar cane,corrected_yield_Wheat,hectares_Alfalfa for forage,hectares_Maize (incl. 0067 and 0068),"hectares_Rice, paddy",hectares_Sugar cane,hectares_Wheat
area_map_name,,,,,,,,,,,,
أبو المنجا,6.036917,4.914650e+08,2.595179e+08,69835693.0,306889760.0,4.181459e+06,38148380.0,6494.87840,14252.12300,63454.140,4.465074e+01,6321.710
أسوان,3.914534,5.400515e+08,2.997159e+08,30603171.0,155922200.0,2.077367e+06,293570370.0,4456.81200,6070.18010,19758.865,2.041726e+01,41938.630
ابوحمار,2.411044,1.732788e+08,2.078340e+07,3607798.5,0.0,3.643663e+09,12218493.0,269.99698,635.26044,0.000,3.999108e+04,1745.499
الابراهيمية,3.099258,3.832278e+09,2.940824e+09,648056190.0,77412970.0,8.464977e-03,741813400.0,37520.32030,121317.04900,10194.049,8.601590e-08,105973.340
الباجورية,3.197584,1.366886e+09,1.125376e+09,62153391.0,19131112.0,3.253317e+06,193613740.0,13221.90582,11657.73063,2210.689,3.143160e+01,27659.105


In [127]:
excel_df = excel_df.merge(
    result_production,
    left_on="command_area_name",
    right_on="area_map_name",
    how="left"
)

excel_df.head()

,water_productivity,surface_water_mean,ground_water_mean,supply_demand_ratio,water_supply,water_demand,gdp_pc,rwi_median,precipitation,potentialevapotransporation,...,corrected_yield_Alfalfa for forage,corrected_yield_Maize (incl. 0067 and 0068),"corrected_yield_Rice, paddy",corrected_yield_Sugar cane,corrected_yield_Wheat,hectares_Alfalfa for forage,hectares_Maize (incl. 0067 and 0068),"hectares_Rice, paddy",hectares_Sugar cane,hectares_Wheat
0,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,2.983641e+08,2.630364e+07,1.585034e-04,0.000002,66391144.0,3525.5088,5108.2285,1.877275e-08,2.402912e-11,9484.449
1,0.000004,0.000838,0.000074,16.210424,6591.709218,6591.715237,10586.715591,0.2365,1684.303418,17488.37100,...,2.983641e+08,2.630364e+07,1.585034e-04,0.000002,66391144.0,3525.5088,5108.2285,1.877275e-08,2.402912e-11,9484.449
2,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,1.033145e+09,1.184452e+08,2.635205e+07,0.011866,238904400.0,14250.7399,20356.6573,3.624428e+03,1.195065e-07,34129.203
3,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,1.033145e+09,1.184452e+08,2.635205e+07,0.011866,238904400.0,14250.7399,20356.6573,3.624428e+03,1.195065e-07,34129.203
4,0.000003,0.000474,0.000071,31.542032,2748.449999,2748.453114,11533.990996,0.2070,890.821122,20860.75963,...,1.033145e+09,1.184452e+08,2.635205e+07,0.011866,238904400.0,14250.7399,20356.6573,3.624428e+03,1.195065e-07,34129.203


In [128]:
w = "pct_of_governorate_area"
sum_cols = ["arable_km2"] + [ c for c in v if "yield" in c.lower() and "hectares" in c.lower()]

exclude = [w, "pct_of_command_area_area", "intersection_km2"]
v = [c for c in excel_df.select_dtypes("number") if c not in exclude]
avg_cols = [c for c in v if c not in sum_cols]

avg = excel_df.groupby("gov_NAME1_").apply(
    lambda g: g[avg_cols].apply(lambda x: np.average(x, weights=g[w]))
)

total = (
    excel_df[sum_cols]
    .mul(excel_df["pct_of_command_area_area"], axis=0)
    .groupby(excel_df["gov_NAME1_"])
    .sum()
)

result_excel = avg.join(total)
result_excel.to_csv("data.csv")

In [ ]:
corelate_df = female_df.merge(
     result_excel,
    left_on="NAME1_",
    right_on="gov_NAME1_",
    how="left"
)

col_normalize = [
    col for col in corelate_df.columns
    if any(word in col.lower() for word in ["corrected", "hectares", "arable"])
]

corelate_df[col_normalize] = corelate_df[col_normalize].div(
    corelate_df["rural_population"],
    axis=0
)

corelate_df = corelate_df.drop(
    columns=["pweight", "numwrksc"],
    errors="ignore"
)

corelate_df .head()



NameError: name 'corelate' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pearson correlation for all numeric variables
corr = corelate_df .select_dtypes("number").corr(method="pearson")

plt.figure(figsize=(20, 16))
sns.heatmap(
    corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    cbar_kws={"label": "Pearson correlation"}
)

plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

corr.to_csv("correlation.csv")